# 03 - Avaliacao

Grade comparativa, CLIPScore, checagem de memorizacao e prep do formulario cego.

## 1. Instalar dependencias

In [ ]:
!pip -q install diffusers transformers accelerate peft torchmetrics matplotlib
!pip -q install -U torchao

## 1.5 Clonar o dataset

In [ ]:
!git clone https://github.com/lramosc1512/atelie-generativo-LeonidIA.git
%cd /content

## 2.5 Login HF

In [ ]:
from huggingface_hub import login
login()

## 2. Carregar o modelo base

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
pipe.set_progress_bar_config(disable=True)
print("Modelo base carregado.")

## 3. Prompts fixos

6 prompts, mesma seed em todos os modelos pra isolar o efeito do LoRA.

In [ ]:
PROMPTS_TESTE = [
    "estilo_brasilia, fachada curva de concreto branco do Congresso Nacional ao entardecer",
    "estilo_brasilia, Catedral Metropolitana com coroa de colunas brancas contra o ceu azul",
    "estilo_brasilia, Palacio do Planalto com colunas curvas e espelho d'agua ao entardecer",
    "estilo_brasilia, Memorial JK com estatua sobre pilar curvo de concreto",
    "estilo_brasilia, Teatro Nacional com painel de relevos cubicos de concreto",
    "estilo_brasilia, Ponte JK com arcos brancos entrelacados sobre o lago ao por do sol",
]

SEED = 42
REPO_CONFIG1 = "lramosc1512/lora-estilo-brasilia-config1"
REPO_CONFIG2 = "lramosc1512/lora-estilo-brasilia-config2"

## 4. Gerar as imagens

Base, config1 e config2 - precisa dos 2 LoRAs já no Hub.

In [ ]:
imagens_geradas = {}  # {(modelo, prompt_idx): imagem_PIL}

def gerar_lote(nome_modelo):
    for i, prompt in enumerate(PROMPTS_TESTE):
        gerador = torch.Generator(device=device).manual_seed(SEED)
        imagem = pipe(prompt, negative_prompt="desfocado, deformado",
                      guidance_scale=7.5, num_inference_steps=30,
                      generator=gerador).images[0]
        imagens_geradas[(nome_modelo, i)] = imagem
        print(f"{nome_modelo} - prompt {i+1}/6 gerado")

# Base (sem LoRA)
gerar_lote("base")

# Config 1
pipe.load_lora_weights(REPO_CONFIG1)
gerar_lote("config1")
pipe.unload_lora_weights()

# Config 2
pipe.load_lora_weights(REPO_CONFIG2)
gerar_lote("config2")
pipe.unload_lora_weights()

print("Geracao concluida:", len(imagens_geradas), "imagens")

## 5. Grade comparativa 6x3

In [ ]:
import matplotlib.pyplot as plt

modelos = ["base", "config1", "config2"]
fig, eixos = plt.subplots(len(PROMPTS_TESTE), len(modelos), figsize=(12, 24))

for linha, prompt in enumerate(PROMPTS_TESTE):
    for coluna, modelo in enumerate(modelos):
        eixo = eixos[linha][coluna]
        eixo.imshow(imagens_geradas[(modelo, linha)])
        eixo.axis("off")
        if linha == 0:
            eixo.set_title(modelo)

plt.tight_layout()
plt.savefig("grade_comparativa.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo em grade_comparativa.png")

## 6. CLIPScore

Alinhamento imagem-texto, não mede fidelidade ao estilo em si.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

scores_por_modelo = {"base": [], "config1": [], "config2": []}

for (modelo, idx), imagem in imagens_geradas.items():
    inputs = clip_processor(text=[PROMPTS_TESTE[idx]], images=imagem, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = clip_model(**inputs)
    img_emb = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
    txt_emb = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)
    score = 100 * (img_emb * txt_emb).sum(dim=-1)
    scores_por_modelo[modelo].append(score.item())

for modelo, scores in scores_por_modelo.items():
    media = sum(scores) / len(scores)
    print(f"{modelo}: CLIPScore medio = {media:.2f}  (por prompt: {[round(s,1) for s in scores]})")

## 7. Verificação de memorização

Compara geração com legenda de treino vs imagem original. Troca MODELO_PARA_TESTAR pra checar o outro config.

In [ ]:
import random
from pathlib import Path
from PIL import Image

MODELO_PARA_TESTAR = REPO_CONFIG2  # troque para REPO_CONFIG1 pra testar o outro

with open("/content/atelie-generativo-LeonidIA/dados/legendas.txt", "r", encoding="utf-8") as f:
    linhas = [l.strip() for l in f if l.strip()]

random.seed(7)
amostra = random.sample(linhas, 10)

pipe.load_lora_weights(MODELO_PARA_TESTAR)

fig, eixos = plt.subplots(10, 2, figsize=(6, 30))
for i, linha in enumerate(amostra):
    arquivo, legenda = linha.split(",", 1)
    legenda = legenda.strip()

    original = Image.open(f"/content/atelie-generativo-LeonidIA/dados/imagens/{arquivo}")
    gerada = pipe(legenda, num_inference_steps=30, guidance_scale=7.5).images[0]

    eixos[i][0].imshow(original)
    eixos[i][0].set_title("original" if i == 0 else "")
    eixos[i][0].axis("off")
    eixos[i][1].imshow(gerada)
    eixos[i][1].set_title("gerada" if i == 0 else "")
    eixos[i][1].axis("off")

pipe.unload_lora_weights()
plt.tight_layout()
plt.savefig("verificacao_memorizacao.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Formulário cego

Embaralha as 18 imagens com nomes genéricos + salva a chave de resposta separada.

In [ ]:
import shutil
import csv
import random

PASTA_CEGA = Path("/content/imagens_formulario_cego")
PASTA_CEGA.mkdir(exist_ok=True)

itens = list(imagens_geradas.items())
random.seed(123)
random.shuffle(itens)

chave_resposta = []
for i, ((modelo, idx), imagem) in enumerate(itens, start=1):
    nome_anonimo = f"imagem_{i:02d}.png"
    imagem.save(PASTA_CEGA / nome_anonimo)
    chave_resposta.append({
        "arquivo": nome_anonimo,
        "modelo": modelo,
        "prompt": PROMPTS_TESTE[idx],
    })

with open(PASTA_CEGA / "CHAVE_RESPOSTA_NAO_COMPARTILHAR.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["arquivo", "modelo", "prompt"])
    writer.writeheader()
    writer.writerows(chave_resposta)

shutil.make_archive("/content/imagens_formulario_cego", "zip", PASTA_CEGA)

from google.colab import files
files.download("/content/imagens_formulario_cego.zip")
print("Baixe o zip - contem as 18 imagens anonimas + a chave de resposta (nao suba a chave no formulario!)")

## 9. Montar o formulário

Manual, fora do Colab - Forms não tem integração automática.

## Depois disso

- Grade, CLIPScore e memorização já rodam sozinhos
- Manda o formulário cego assim que tiver as 18 imagens
- Guarda os pngs pro relatório

```bash
git add notebooks/03_avaliacao.ipynb
git commit -m "notebook avaliacao"
git push
```